# Tokens et logits — imports

Charge PyTorch et le modèle du projet.

In [1]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model

# Tokenisation

Transforme le texte en tenseurs exploitables par le modèle.

In [2]:
llm = load_model()

tokenizer = llm.tokenizer
model = llm.model
device = llm.device

text = "Paris est la capitale de"

encoded = tokenizer(
    text,
    return_tensors="pt",
)

encoded

Chargement du modèle : Qwen/Qwen2.5-3B-Instruct
Device utilisé : cuda


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

C:\Users\emine\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\emine\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

C:\Users\emine\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\emine\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'input_ids': tensor([[59604,  1788,  1187, 60410,  1574,   409]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

# IDs des tokens

Affiche les identifiants produits par le tokenizer.

In [3]:
input_ids = encoded["input_ids"]

print("Shape :", input_ids.shape)
print("IDs :", input_ids)

Shape : torch.Size([1, 6])
IDs : tensor([[59604,  1788,  1187, 60410,  1574,   409]])


# Tokens réels

Associe chaque identifiant à son token.

In [4]:
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

for token_id, token in zip(input_ids[0].tolist(), tokens):
    print(f"{token_id:>8} -> {repr(token)}")

   59604 -> 'Paris'
    1788 -> 'Ġest'
    1187 -> 'Ġla'
   60410 -> 'Ġcapit'
    1574 -> 'ale'
     409 -> 'Ġde'


# Tokens vs mots

Compare le nombre de tokens au nombre de mots.

In [5]:
print("Nombre de tokens :", len(tokens))
print("Nombre de mots approximatif :", len(text.split()))

Nombre de tokens : 6
Nombre de mots approximatif : 5


# Décodage

Reconstruit le texte à partir des IDs.

In [6]:
decoded = tokenizer.decode(input_ids[0])

print("Original :", text)
print("Décodé  :", decoded)

Original : Paris est la capitale de
Décodé  : Paris est la capitale de


# Exemples de tokenisation

Observe le découpage de plusieurs chaînes.

In [7]:
examples = [
    "chat",
    "anticonstitutionnellement",
    "Transformer",
    "🤖",
]

for example in examples:
    ids = tokenizer.encode(example, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print()
    print("Texte :", example)
    print("IDs   :", ids)
    print("Tokens:", tokens)


Texte : chat
IDs   : [9686]
Tokens: ['chat']

Texte : anticonstitutionnellement
IDs   : [517, 1924, 10446, 8138, 986]
Tokens: ['ant', 'icon', 'stitution', 'nel', 'lement']

Texte : Transformer
IDs   : [46358]
Tokens: ['Transformer']

Texte : 🤖
IDs   : [147238]
Tokens: ['ðŁ¤ĸ']


# Transfert sur le device

Place les tenseurs sur CPU ou GPU.

In [8]:
inputs = {
    key: value.to(device)
    for key, value in encoded.items()
}

# Forward pass

Fait passer l’entrée dans le Transformer.

In [9]:
with torch.no_grad():
    outputs = model(**inputs)

# Logits

Observe la forme de la sortie du modèle.

In [10]:
logits = outputs.logits

print(logits.shape)

torch.Size([1, 6, 151936])


# Dernière position

Récupère les logits utilisés pour prédire le prochain token.

In [11]:
last_token_logits = logits[:, -1, :]

print(last_token_logits.shape)
print(last_token_logits)

torch.Size([1, 151936])
tensor([[ 5.8438,  6.2812,  4.9688,  ..., -4.0312, -4.0312, -4.0312]],
       device='cuda:0', dtype=torch.bfloat16)


# Softmax

Convertit les logits en probabilités.

In [12]:
probabilities = torch.softmax(
    last_token_logits,
    dim=-1,
)

print(probabilities.shape)
print(probabilities.sum())

torch.Size([1, 151936])
tensor(1., device='cuda:0', dtype=torch.bfloat16)


# Top 10

Affiche les tokens les plus probables.

In [13]:
top_k = 10

top_probs, top_ids = torch.topk(
    probabilities[0],
    k=top_k,
)

for probability, token_id in zip(
    top_probs.tolist(),
    top_ids.tolist(),
):
    token = tokenizer.decode([token_id])

    print(
        f"{repr(token):20s} "
        f"id={token_id:<8} "
        f"probabilité={probability:.4f}"
    )

' la'                id=1187     probabilité=0.7812
' l'                 id=326      probabilité=0.1055
' France'            id=9625     probabilité=0.0388
' quel'              id=25025    probabilité=0.0052
' ('                 id=320      probabilité=0.0052
' :'                 id=549      probabilité=0.0049
' qui'               id=7774     probabilité=0.0049
' notre'             id=28349    probabilité=0.0046
' '                  id=220      probabilité=0.0041
' ______'            id=32671    probabilité=0.0025


# Argmax

Sélectionne le token le plus probable.

In [14]:
next_token_id = torch.argmax(
    probabilities,
    dim=-1,
)

print("ID :", next_token_id.item())

next_token = tokenizer.decode(
    next_token_id
)

print("Token choisi :", repr(next_token))

ID : 1187
Token choisi : ' la'


# Vérification

Confirme que logits et softmax donnent le même argmax.

In [15]:
next_token_from_logits = torch.argmax(
    last_token_logits,
    dim=-1,
)

assert next_token_id.item() == next_token_from_logits.item()

# Ajout d’un token

Ajoute manuellement le token prédit à la séquence.

In [16]:
new_input_ids = torch.cat(
    [
        inputs["input_ids"],
        next_token_id.unsqueeze(0),
    ],
    dim=-1,
)

print(
    tokenizer.decode(
        new_input_ids[0],
        skip_special_tokens=True,
    )
)

Paris est la capitale de la


# Greedy decoding

Teste la génération gloutonne implémentée dans le projet.

In [17]:
from src.llm.decoding import greedy_generate

result = greedy_generate(
    model=model,
    tokenizer=tokenizer,
    prompt="Paris est la capitale de",
    device=device,
    max_new_tokens=20,
)

print(result)

Paris est la capitale de la France. C'est une ville très grande. Elle est très belle. Elle a beaucoup de mus


# `model.generate()`

Compare avec la génération native de Transformers.

In [18]:
encoded = tokenizer(
    "Paris est la capitale de",
    return_tensors="pt",
).to(device)

with torch.no_grad():

    generated_ids = model.generate(
        **encoded,
        max_new_tokens=20,
        do_sample=False,
    )

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )
)

Paris est la capitale de la France. C'est une ville très grande et très peuplée. Elle est située dans
